# 📱 Pocket OTC AI Analyzer — APK Builder

هذه النسخة تبني APK مباشرة من GitHub داخل Google Colab، ولا تستخدم Telegram أو GitHub Actions.

**READ-ONLY:** التطبيق لا يسجل الدخول إلى Pocket Option ولا ينفذ صفقات.

In [ ]:
# 🚀 بناء APK بنقرة واحدة
import os, shutil, subprocess, time, traceback, zipfile

REPO = 'https://github.com/mohmb142/Jjjjjjj.git'
ZIP_URL = 'https://github.com/mohmb142/Jjjjjjj/archive/refs/heads/main.zip'
ROOT = '/content/Jjjjjjj'

try:
    if os.path.exists(ROOT):
        shutil.rmtree(ROOT, ignore_errors=True)

    print('1/5 ⬇️ تنزيل المشروع...')
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO, ROOT], check=True, timeout=120)
    except Exception:
        print('⚠️ تعذر git clone، سيتم استخدام ZIP...')
        zip_path = '/content/Jjjjjjj.zip'
        subprocess.run(['wget', '-q', '-O', zip_path, ZIP_URL], check=True, timeout=120)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall('/content')
        extracted = '/content/Jjjjjjj-main'
        if not os.path.isdir(extracted):
            raise RuntimeError('تعذر استخراج نسخة GitHub')
        os.rename(extracted, ROOT)

    android_dir = os.path.join(ROOT, 'phone-agent')
    gradle_file = os.path.join(android_dir, 'app', 'build.gradle.kts')
    if not os.path.isfile(gradle_file):
        raise RuntimeError('مشروع Android غير موجود في phone-agent/')

    print('2/5 ☕ تجهيز Java وGradle...')
    subprocess.run(['apt-get', 'update', '-qq'], check=True, timeout=180)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jdk', 'wget', 'unzip'], check=True, timeout=180)
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
    os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

    gradle_bin = '/content/gradle-8.7/bin/gradle'
    if not os.path.exists(gradle_bin):
        subprocess.run(['wget', '-q', 'https://services.gradle.org/distributions/gradle-8.7-bin.zip', '-O', '/content/gradle.zip'], check=True, timeout=180)
        with zipfile.ZipFile('/content/gradle.zip') as z:
            z.extractall('/content')

    print('3/5 🔨 بناء APK...')
    subprocess.run([gradle_bin, 'assembleDebug', '--no-daemon', '--stacktrace'], cwd=android_dir, check=True, timeout=900)

    apk = os.path.join(android_dir, 'app', 'build', 'outputs', 'apk', 'debug', 'app-debug.apk')
    if not os.path.isfile(apk) or os.path.getsize(apk) == 0:
        raise RuntimeError('لم يتم إنشاء APK')

    print('4/5 🧪 التحقق من ملف APK...')
    print(f'حجم APK: {os.path.getsize(apk) / 1024 / 1024:.2f} MB')

    print('5/5 📦 تجهيز الملف للتحميل...')
    from google.colab import files
    files.download(apk)
    print('✅ تم إنشاء Pocket OTC AI Analyzer APK بنجاح.')
    print('📱 التطبيق يدعم Android 8+، وبالتالي Android 10.')
    print('🔒 بدون Telegram وبدون تنفيذ صفقات.')

except Exception:
    print('❌ فشل البناء:')
    traceback.print_exc()
    print('إذا فشل apt أو Gradle، أعد تشغيل جلسة Colab ثم شغّل الخلية مرة أخرى.')